In [ ]:
import sys
import os

# Get the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

# Add parent directory to sys.path
sys.path.append(parent_dir)

print(parent_dir)


In [ ]:
import torch
import torch.optim as optim
from itables import init_notebook_mode, show
import numpy as np
import importlib

import deeparguing.gradual_aacbr as gradual_aacbr
import deeparguing.semantics.mlp_semantics as ms
import deeparguing.base_scores.feature_weighted_base_score as fwbs
import deeparguing.casebase_edge_weights.feature_weighted_partial_order as fwpo
import deeparguing.irrelevance_edge_weights.regular_irrelevance as ri

from deeparguing.train import dynamic_train_model, evaluate_model

from helper import load_iris, split_data

init_notebook_mode(all_interactive=True)

In [ ]:
def reload_imports():
    importlib.reload(gradual_aacbr)
    importlib.reload(ms)
    importlib.reload(fwbs)
    importlib.reload(fwpo)
    importlib.reload(ri)

reload_imports()

In [ ]:
SEED = 42

## Data Set

In [ ]:
X, y = load_iris()
show(X)
show(y)

In [ ]:
all_y = np.unique(y, axis=0)

## Train Model

### Split into Training, Validation and Test

In [ ]:
X, y = torch.tensor(X), torch.tensor(y, dtype=torch.float32)

train_full, train, val, test = split_data(X, y, SEED)

print(f"Test Size:  {len(test['X'])}")
print(f"Train Size:  {len(train['X'])}")
print(f"Validation Size:  {len(val['X'])}")

### Train Model

In [ ]:
DEFAULT_CASE = train["X"].mean(axis=0)

X_DEFAULTS = DEFAULT_CASE.tile(len(all_y), 1)
Y_DEFAULTS = torch.tensor(all_y)

MAX_ITERS = 1 
EPOCHS = 75 
N_SPLITS = 5
USE_SYMMETRIC_ATTACKS = True

In [ ]:
reload_imports()
torch.manual_seed(0) # TRY DIFFERENT INITIAL WEIGHTS 

no_features = train["X"].shape[-1]
semantics = ms.MLPBasedSemantics(max_iters=MAX_ITERS, epsilon=0)
partial_order = fwpo.FeatureWeightedPartialOrder(no_features, sharpness=100)
irrelevance = ri.RegularIrrelevance(partial_order)
base_score = fwbs.FeatureWeightedBaseScore(no_features)

model = gradual_aacbr.GradualAACBR(semantics, 
                                base_score,
                                irrelevance,
                                partial_order)

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

In [ ]:
reload_imports()
evaluate_model(model, train["X"], train["y"], X_DEFAULTS, Y_DEFAULTS, val["X"], val["y"], show_confusion=True,  print_matrix=True, print_compute_graph=True, print_graph = False )
model.plot_casebase_edge_weights_parameters()

In [ ]:

DISABLE_TQDM = False

In [ ]:
dynamic_train_model(model, train["X"], train["y"], X_DEFAULTS, Y_DEFAULTS, optimizer, criterion, EPOCHS, use_symmetric_attacks=True, n_splits = N_SPLITS, use_blockers=False, random_split_state=42, plot_loss_curve=True)

In [ ]:
reload_imports()
with torch.no_grad():
    evaluate_model(model, train["X"], train["y"], X_DEFAULTS, Y_DEFAULTS, val["X"], val["y"], show_confusion=True,  print_matrix=True, print_compute_graph=False  )

model.plot_casebase_edge_weights_parameters()

### Test Set

In [ ]:
reload_imports()
with torch.no_grad():
    evaluate_model(model, train_full["X"], train_full["y"], X_DEFAULTS, Y_DEFAULTS, test["X"], test["y"], show_confusion=True)